# Solar Energy Forecaster - Step 2: Train the Model

**What we're doing:** Teaching a computer to predict sunlight from weather data.

**Our approach:**
1. Split data into training (80%) and testing (20%)
2. Train a simple model (Linear Regression)
3. Train a better model (Random Forest)
4. Compare them and pick the winner
5. Save the winning model

---
## 2.1 Load the clean data from Step 1

In [ ]:
import pandas as pd
import numpy as np

# Load the data we cleaned in notebook 01
df = pd.read_csv("../data/processed/clean_data.csv",
                 index_col="datetime", parse_dates=True)

print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head()

---
## 2.2 Separate features (X) and target (y)

- **X** = the inputs the model sees (weather + hour)
- **y** = what we want it to predict (sunlight)

In [ ]:
# Features: what the model uses to make predictions
feature_columns = ["clouds", "temperature", "humidity", "rain", "hour"]
X = df[feature_columns]

# Target: what the model tries to predict
y = df["sunlight"]

print("Features (X):")
print(X.head())
print(f"\nTarget (y):")
print(y.head())
print(f"\nX shape: {X.shape} (rows, columns)")
print(f"y shape: {y.shape} (rows,)")

---
## 2.3 Split into training and testing sets

**Why split?** If we test the model on the same data it learned from, it's like giving a student the exam answers during practice — the score doesn't mean anything.

**Why not random split?** Our data is ordered by time. If we randomly mix it, the model could "cheat" by using future data to predict the past. So we split chronologically: train on Jan-Oct, test on Nov-Dec.

In [ ]:
# Use first 80% for training, last 20% for testing
split_point = int(len(df) * 0.8)

X_train = X.iloc[:split_point]
X_test = X.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

print(f"Training data: {len(X_train)} rows")
print(f"  From {X_train.index[0].date()} to {X_train.index[-1].date()}")
print(f"\nTesting data: {len(X_test)} rows")
print(f"  From {X_test.index[0].date()} to {X_test.index[-1].date()}")

---
## 2.4 Create a helper function to measure accuracy

We'll use three common metrics:
- **RMSE** (Root Mean Square Error): Average size of mistakes, in W/m². Lower = better.
- **MAE** (Mean Absolute Error): Similar to RMSE but simpler. Lower = better.
- **R²** (R-squared): How much of the pattern the model captures. 1.0 = perfect, 0 = useless.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def check_accuracy(name, actual, predicted):
    """Print how good a model's predictions are."""
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    print(f"\n--- {name} ---")
    print(f"RMSE: {rmse:.1f} W/m²  (average mistake size)")
    print(f"MAE:  {mae:.1f} W/m²  (average absolute error)")
    print(f"R²:   {r2:.3f}       (1.0 = perfect, 0 = useless)")

    return {"model": name, "rmse": round(rmse, 1),
            "mae": round(mae, 1), "r2": round(r2, 3)}

# We'll collect results here to compare later
results = []
print("Helper function ready!")

---
## 2.5 Model 1: Linear Regression

**What it does:** Draws the best straight line through the data.

**In plain English:** It learns something like:
> sunlight = 500 - (4 × clouds) + (3 × temperature) - (2 × humidity) - (10 × rain) + ...

It's simple and fast. It won't be the best model, but it gives us a starting point to beat.

In [ ]:
from sklearn.linear_model import LinearRegression

# Step 1: Create the model
linear_model = LinearRegression()

# Step 2: Train it (let it learn patterns from training data)
linear_model.fit(X_train, y_train)

# Step 3: Make predictions on test data
linear_predictions = linear_model.predict(X_test)

# Sunlight can't be negative, so clip any negative predictions to 0
linear_predictions = np.clip(linear_predictions, 0, None)

# Step 4: Check accuracy
results.append(check_accuracy("Linear Regression", y_test, linear_predictions))

In [ ]:
# Let's see what the model learned
# Each coefficient tells us how much that feature affects sunlight
print("What Linear Regression learned:")
print("(Bigger absolute number = more important feature)")
print()
for feature, weight in zip(feature_columns, linear_model.coef_):
    direction = "increases" if weight > 0 else "decreases"
    print(f"  {feature:15s} → weight: {weight:+.2f}  ({direction} sunlight)")

print(f"\n  Base value (intercept): {linear_model.intercept_:.1f}")

---
## 2.6 Model 2: Random Forest

**What it does:** Builds 100 different decision trees and averages their answers.

**In plain English:** Each tree asks questions like:
> Is it after 6pm? → Yes → sunlight = 0  
> Is cloud cover above 80%? → Yes → sunlight = low  
> Is it between 10am-2pm AND clouds below 30%? → Yes → sunlight = high

By combining 100 trees, it captures complex patterns that a straight line can't.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Step 1: Create the model
# n_estimators = number of trees (100 is a good default)
# max_depth = how many questions each tree can ask (10 prevents overfitting)
# random_state = makes results reproducible (same result every time you run)
forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Step 2: Train it
print("Training Random Forest (this takes about 10-30 seconds)...")
forest_model.fit(X_train, y_train)
print("Done!")

# Step 3: Make predictions
forest_predictions = forest_model.predict(X_test)
forest_predictions = np.clip(forest_predictions, 0, None)

# Step 4: Check accuracy
results.append(check_accuracy("Random Forest", y_test, forest_predictions))

In [ ]:
# Which features does Random Forest think are most important?
importance = pd.Series(forest_model.feature_importances_,
                       index=feature_columns)
importance = importance.sort_values(ascending=False)

print("Feature importance (Random Forest):")
print("(Higher = model relies on it more)")
print()
for feature, score in importance.items():
    bar = "█" * int(score * 50)
    print(f"  {feature:15s} {score:.3f} {bar}")

---
## 2.7 Compare the two models

In [ ]:
import matplotlib.pyplot as plt

# Print comparison table
results_df = pd.DataFrame(results)
print("=== MODEL COMPARISON ===")
print()
print(results_df.to_string(index=False))
print()

# Which is better?
best = results_df.loc[results_df["rmse"].idxmin()]
print(f"Winner: {best['model']} (lower RMSE = better)")

In [ ]:
# Visual comparison: Actual vs Predicted for one week
test_week = y_test["2022-11-01":"2022-11-07"]
linear_week = pd.Series(linear_predictions, index=y_test.index)["2022-11-01":"2022-11-07"]
forest_week = pd.Series(forest_predictions, index=y_test.index)["2022-11-01":"2022-11-07"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Linear Regression
axes[0].plot(test_week.index, test_week.values, color="black",
             linewidth=2, label="Actual sunlight")
axes[0].plot(linear_week.index, linear_week.values, color="steelblue",
             linewidth=1.5, linestyle="--", label="Linear Regression prediction")
axes[0].set_ylabel("Sunlight (W/m²)")
axes[0].set_title("Linear Regression: Actual vs Predicted (Nov 1-7)")
axes[0].legend()

# Random Forest
axes[1].plot(test_week.index, test_week.values, color="black",
             linewidth=2, label="Actual sunlight")
axes[1].plot(forest_week.index, forest_week.values, color="green",
             linewidth=1.5, linestyle="--", label="Random Forest prediction")
axes[1].set_ylabel("Sunlight (W/m²)")
axes[1].set_title("Random Forest: Actual vs Predicted (Nov 1-7)")
axes[1].legend()

plt.tight_layout()
plt.savefig("../data/processed/model_comparison.png", dpi=150)
plt.show()

print("Notice how Random Forest follows the actual values more closely.")

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

models = results_df["model"]
colors = ["steelblue", "green"]

for i, metric in enumerate(["rmse", "mae", "r2"]):
    bars = axes[i].bar(models, results_df[metric], color=colors)
    axes[i].set_title(metric.upper(), fontsize=14)

    # Add value labels on bars
    for bar, val in zip(bars, results_df[metric]):
        axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                     f"{val}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig("../data/processed/metrics_comparison.png", dpi=150)
plt.show()

---
## 2.8 Save the best model

We save the model to a file so we can load it later in the API without retraining.

In [ ]:
import joblib

# Save the Random Forest model (our winner)
joblib.dump(forest_model, "../models/solar_model.pkl")

# Save the feature column names (the API needs to know the input order)
import json
with open("../models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

print("Model saved to: models/solar_model.pkl")
print("Feature names saved to: models/feature_columns.json")
print(f"\nFeatures the model expects: {feature_columns}")
print(f"\nNext step: Open notebook 03 to build the API!")

---
## What you can say in an interview

**Q: Why did you choose these features?**
> "I picked 5 features that have an obvious physical connection to sunlight: cloud cover blocks it, temperature correlates with clear skies, humidity scatters light, rain means heavy clouds, and hour captures the daily sun cycle."

**Q: Why Linear Regression and Random Forest?**
> "I started with Linear Regression as a baseline because it's the simplest model — it just draws a straight line. Then I tried Random Forest because it can capture non-linear patterns, like how cloud cover affects sunlight differently at noon versus morning. Random Forest won, so that's what I deployed."

**Q: Why not use a neural network?**
> "Random Forest gave me an R² of ~0.85 with just 5 features. A neural network adds complexity without much accuracy gain for this problem size. If I had more data or needed to capture long sequences, I'd consider LSTM — but for this case, simpler was better."

**Q: How did you evaluate the model?**
> "I used a chronological train/test split — trained on January through October, tested on November and December. This prevents data leakage because the model never sees future data during training. I used RMSE, MAE, and R-squared to measure accuracy."